In [ ]:
import numpy as np
import pandas as pd
import ast
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# Load datasets
movie = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv("data/tmdb_5000_credits.csv")

# Merge both datasets on title
movies = movie.merge(credits, on='title')

# Keep important columns
movies = movies[['id', 'title', 'overview', 'keywords', 'genres', 'cast', 'crew']]

# Drop rows with null values
movies.dropna(inplace=True)

In [ ]:
# Process 'overview' column
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# Process 'keywords' column
def convert_keywords(obj):
    l = []
    for i in ast.literal_eval(obj):
        l.append(i['name'])
    return l

movies['keywords'] = movies['keywords'].apply(convert_keywords)

In [ ]:
# Process 'genres' column
def convert_genres(obj):
    l = []
    for i in ast.literal_eval(obj):
        l.append(i['name'])
    return l

movies['genres'] = movies['genres'].apply(convert_genres)

In [ ]:
# Process 'cast' column (top 3 only)
def extract_cast(obj):
    l = []
    count = 0
    for i in ast.literal_eval(obj):
        if count != 3:
            l.append(i['name'])
            count += 1
        else:
            break
    return l

movies['cast'] = movies['cast'].apply(extract_cast)

In [ ]:
# Process 'crew' column (get director only)
def extract_director(obj):
    l = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            l.append(i['name'])
            break
    return l

movies['crew'] = movies['crew'].apply(extract_director)

# Create 'tags' column by combining overview + keywords + genres + cast + crew
movies['tags'] = movies['overview'] + movies['cast'] + movies['crew'] + movies['keywords']

# Final dataset with relevant columns
movies = movies[['id', 'title', 'tags']]

In [ ]:
# Remove spaces from tags
movies['tags'] = movies['tags'].apply(lambda x: [i.replace(" ", "") for i in x])

# Stemming
ps = PorterStemmer()

def stemming(text):
    l = []
    for i in text:
        l.append(ps.stem(i))
    return " ".join(l)

movies['tags'] = movies['tags'].apply(stemming)

In [ ]:
# Vectorization
vectorizer = CountVectorizer(max_features=500, stop_words='english')
vectors = vectorizer.fit_transform(movies['tags']).toarray()

# Cosine similarity
similarity = cosine_similarity(vectors)

In [ ]:
# Recommendation function
def Recommendation_system(movie_title):
    movie_index = movies[movies['title'] == movie_title].index[0]
    distances = sorted(list(enumerate(similarity[movie_index])), reverse=True, key=lambda x: x[1])
    
    for i in distances[1:20]:
        print(movies.iloc[i[0]].title)

# Example usage
# Recommendation_system('Avatar')

In [ ]:
# Save the model and data
pickle.dump(movies, open('model.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

## Bonus: User-Based Collaborative Filtering

The TMDB dataset does not include user ratings, so we use the **MovieLens 100K** dataset (built into the `surprise` library) to demonstrate collaborative filtering. We train an **SVD** model to predict user ratings for unseen movies.

In [ ]:
from surprise import Dataset, SVD, accuracy
from surprise.model_selection import train_test_split

# Load built-in MovieLens 100K dataset
data = Dataset.load_builtin('ml-100k')
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Train SVD model
svd = SVD(n_factors=100, random_state=42)
svd.fit(trainset)

# Evaluate on test set
predictions = svd.test(testset)
print("Collaborative Filtering (SVD) RMSE:")
accuracy.rmse(predictions)

In [ ]:
# Demonstrate collaborative filtering recommendations for a sample user
user_id = '196'

# Get movies already rated by this user
inner_uid = svd.trainset.to_inner_uid(user_id)
rated_by_user = set(
    svd.trainset.to_raw_iid(i) for i, _ in svd.trainset.ur[inner_uid]
)

# Get all movie IDs and find unseen ones
all_item_ids = [svd.trainset.to_raw_iid(i) for i in svd.trainset.all_items()]
unseen = [iid for iid in all_item_ids if iid not in rated_by_user]

# Predict ratings for unseen movies and sort
cf_predictions = [(iid, svd.predict(user_id, iid).est) for iid in unseen]
cf_predictions.sort(key=lambda x: x[1], reverse=True)

print(f"Top 10 Collaborative Filtering Recommendations for User {user_id}:")
print(f"{'Movie ID':<12} {'Predicted Rating'}")
print("-" * 30)
for iid, est in cf_predictions[:10]:
    print(f"{iid:<12} {est:.2f}")

## Bonus: Hybrid Recommender

A hybrid recommender combines content-based and collaborative filtering scores using a weighted average:

**hybrid_score = α × content_similarity + (1 - α) × normalized_cf_rating**

Since the TMDB (content-based) and MovieLens (collaborative) datasets use different movie ID systems, we demonstrate the concept below using content-based scores with simulated CF scores. In a production system, a movie ID mapping would bridge both datasets.

In [ ]:
# Hybrid Recommender Demonstration
alpha = 0.6  # weight for content-based scores

# Get content-based similarity scores for a selected movie
movie_title = 'Avatar'
movie_index = movies[movies['title'] == movie_title].index[0]
content_scores = list(enumerate(similarity[movie_index]))
content_scores = sorted(content_scores, key=lambda x: x[1], reverse=True)[1:11]

print(f"Hybrid Recommendations for '{movie_title}' (alpha={alpha}):")
print(f"{'Title':<40} {'Content':<10} {'CF (sim.)':<10} {'Hybrid'}")
print("-" * 75)

np.random.seed(42)
for idx, content_sim in content_scores:
    # In a full implementation, this CF score would come from the SVD model
    # using a movie ID mapping between TMDB and MovieLens datasets.
    # Here we use a simulated normalized CF score for demonstration.
    simulated_cf_score = np.random.uniform(0.5, 1.0)
    hybrid_score = alpha * content_sim + (1 - alpha) * simulated_cf_score
    print(f"{movies.iloc[idx]['title']:<40} {content_sim:<10.4f} {simulated_cf_score:<10.4f} {hybrid_score:.4f}")

print("\nNote: CF scores are simulated because TMDB and MovieLens use different")
print("movie ID systems. A production hybrid system would require an ID mapping.")